# Trexquant 4-Phase Cross-Sectional Alpha Engine

**Objective:** Production-grade Kaggle notebook for cross-sectional return prediction with IC optimization.

Phases:
1. Contrastive Denoising Autoencoder (DAE)
2. Per-date Feature Neutralization (Risk Stripping)
3. Cross-Sectional Transformer with Pairwise LTR
4. Meta-Ensemble with LightGBM + Transformer + SLSQP blend


In [ ]:
# ============================================================
# 0) Setup / Config
# ============================================================
import os
import gc
import random
import warnings
import numpy as np
import pandas as pd
import scipy.linalg
from scipy.optimize import minimize
from scipy import sparse
import lightgbm as lgb
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PIN_MEMORY = DEVICE.type == 'cuda'
NUM_WORKERS = 2

FAST_MODE = True
N_FOLDS = 3 if FAST_MODE else 5
EMBARGO = 5

DAE_EPOCHS = 5
DAE_BATCH_SIZE = 8192
DAE_LR = 1e-3
MASK_RATIO = 0.30

TRF_EPOCHS = 4 if FAST_MODE else 8
TRF_LR = 2e-4
TRF_WD = 1e-4
TRF_D_MODEL = 64
TRF_HEADS = 4
TRF_LAYERS = 2
TRF_DROPOUT = 0.1
PAIR_MAX = 5000

TRAIN_PATH = '/kaggle/input/competitions/earnings-return-prediction-challenge-2025-q-4/train.csv'
TEST_PATH = '/kaggle/input/competitions/earnings-return-prediction-challenge-2025-q-4/test.csv'
SUB_PATH = 'submission.csv'


def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def feature_cols(df):
    return [f'f_{i}' for i in range(1, 173) if f'f_{i}' in df.columns]


def date_ic(y_true, y_pred, di):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    di = np.asarray(di)
    out = []
    for d in np.unique(di):
        m = di == d
        yt = y_true[m]
        yp = y_pred[m]
        if yt.size < 3:
            continue
        s1 = yt.std()
        s2 = yp.std()
        if s1 < 1e-12 or s2 < 1e-12:
            continue
        out.append(np.corrcoef(yt, yp)[0, 1])
    return float(np.mean(out)) if out else 0.0


def make_walkforward_folds(di, n_folds=N_FOLDS, embargo=EMBARGO):
    d = np.sort(np.unique(di))
    folds = []
    for k in range(n_folds):
        v0 = int(len(d) * (k + 1) / (n_folds + 1))
        v1 = int(len(d) * (k + 2) / (n_folds + 1))
        train_dates = d[:max(0, v0 - embargo)]
        val_dates = d[v0:v1]
        if len(train_dates) == 0 or len(val_dates) == 0:
            continue
        folds.append((train_dates, val_dates))
    return folds

print('Device:', DEVICE)


In [ ]:
# ============================================================
# 1) Load data + Median imputation base matrix
# ============================================================
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

F_COLS = feature_cols(train)
assert len(F_COLS) == 172, f'Expected 172 features, found {len(F_COLS)}'
assert 'target' in train.columns
assert all(c in train.columns for c in ['di', 'si', 'sector', 'industry'])
assert all(c in test.columns for c in ['di', 'si', 'sector', 'industry'])

train = train.sort_values('di').reset_index(drop=True)
test = test.sort_values('di').reset_index(drop=True)

X_train_raw = train[F_COLS].to_numpy(np.float32)
X_test_raw = test[F_COLS].to_numpy(np.float32)

X_all = np.vstack([X_train_raw, X_test_raw]).astype(np.float32)
med = np.nanmedian(X_all, axis=0)
med = np.where(np.isfinite(med), med, 0.0).astype(np.float32)

nan_mask = np.isnan(X_all)
if nan_mask.any():
    X_all[nan_mask] = np.take(med, np.where(nan_mask)[1])

n_train = len(train)
X_train_filled = X_all[:n_train]
X_test_filled = X_all[n_train:]

y_train = train['target'].to_numpy(np.float32)
di_train = train['di'].to_numpy(np.int32)
di_test = test['di'].to_numpy(np.int32)

print('train:', train.shape, 'test:', test.shape)
print('X_train_filled:', X_train_filled.shape, 'X_test_filled:', X_test_filled.shape)
cleanup()


In [ ]:
# ============================================================
# 2) PHASE 1: Contrastive Denoising Autoencoder (DAE)
# ============================================================
class MaskedAutoencoder(nn.Module):
    def __init__(self, in_dim=172, hid_dim=64, emb_dim=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, hid_dim),
            nn.GELU(),
            nn.Linear(hid_dim, emb_dim),
            nn.GELU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(emb_dim, hid_dim),
            nn.GELU(),
            nn.Linear(hid_dim, in_dim),
        )

    def forward(self, x, mask_ratio=0.30):
        if self.training and mask_ratio > 0:
            keep = (torch.rand_like(x) > mask_ratio).float()
            x_in = x * keep
        else:
            x_in = x
        z = self.encoder(x_in)
        rec = self.decoder(z)
        return rec, z


def train_dae_and_extract_embeddings(X_all_np):
    model = MaskedAutoencoder(172, 64, 32).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=DAE_LR, weight_decay=1e-5)
    loss_fn = nn.MSELoss()

    ds = TensorDataset(torch.from_numpy(X_all_np))
    dl = DataLoader(
        ds,
        batch_size=DAE_BATCH_SIZE,
        shuffle=True,
        drop_last=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )

    model.train()
    for epoch in tqdm(range(DAE_EPOCHS), desc='DAE epochs'):
        losses = []
        for (xb,) in tqdm(dl, desc=f'DAE train e{epoch+1}', leave=False):
            xb = xb.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            rec, _ = model(xb, mask_ratio=MASK_RATIO)
            loss = loss_fn(rec, xb)
            loss.backward()
            opt.step()
            losses.append(loss.item())
        print(f'epoch={epoch+1} mse={np.mean(losses):.6f}')
        cleanup()

    model.eval()
    emb_chunks = []
    dl_eval = DataLoader(
        ds,
        batch_size=DAE_BATCH_SIZE,
        shuffle=False,
        drop_last=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )
    with torch.no_grad():
        for (xb,) in tqdm(dl_eval, desc='DAE embedding extract'):
            xb = xb.to(DEVICE, non_blocking=True)
            _, z = model(xb, mask_ratio=0.0)
            emb_chunks.append(z.detach().cpu().numpy().astype(np.float32))
    X_emb = np.vstack(emb_chunks)

    del ds, dl, dl_eval, emb_chunks
    cleanup()
    return model, X_emb


dae_model, X_emb_all = train_dae_and_extract_embeddings(X_all)
X_emb_train = X_emb_all[:n_train]
X_emb_test = X_emb_all[n_train:]

print('X_emb_train:', X_emb_train.shape, 'X_emb_test:', X_emb_test.shape)
cleanup()


In [ ]:
# ============================================================
# 3) PHASE 2: Feature Orthogonalization (Risk Stripping)
#    X_neut = X - S (S^T S)^-1 S^T X  (implemented via least squares)
# ============================================================
def build_exposure_matrix(df):
    base = df[['sector', 'industry']].copy()
    base['sector'] = base['sector'].fillna(-1).astype(np.int32)
    base['industry'] = base['industry'].fillna(-1).astype(np.int32)

    cat = pd.get_dummies(
        base,
        columns=['sector', 'industry'],
        prefix=['sec', 'ind'],
        sparse=True,
        dtype=np.float32,
    )
    S_cat = cat.sparse.to_coo().tocsr()

    liq_cols = [c for c in ['top500', 'top1000', 'top2000'] if c in df.columns]
    if liq_cols:
        liq = sparse.csr_matrix(df[liq_cols].fillna(0).to_numpy(np.float32))
        S = sparse.hstack([S_cat, liq], format='csr', dtype=np.float32)
    else:
        S = S_cat.astype(np.float32)

    return S


def neutralize_by_date(X, S_csr, di):
    X = np.asarray(X, dtype=np.float32)
    di = np.asarray(di)
    out = np.empty_like(X, dtype=np.float32)

    for d in tqdm(np.unique(di), desc='Neutralize by date'):
        idx = np.where(di == d)[0]
        Xd = X[idx]
        Sd = S_csr[idx].toarray().astype(np.float32, copy=False)

        if Sd.shape[1] == 0:
            out[idx] = Xd
            continue

        coef = scipy.linalg.lstsq(Sd, Xd, cond=None, check_finite=False, lapack_driver='gelsy')[0]
        out[idx] = Xd - Sd @ coef

    return out


S_train = build_exposure_matrix(train)
S_test = build_exposure_matrix(test)

X_train_stack = np.hstack([X_train_filled, X_emb_train]).astype(np.float32)
X_test_stack = np.hstack([X_test_filled, X_emb_test]).astype(np.float32)

X_train_neut = neutralize_by_date(X_train_stack, S_train, di_train)
X_test_neut = neutralize_by_date(X_test_stack, S_test, di_test)

print('X_train_neut:', X_train_neut.shape, 'X_test_neut:', X_test_neut.shape)
cleanup()


In [ ]:
# ============================================================
# 4) PHASE 3: Cross-Sectional Transformer + Pairwise LTR
# ============================================================
class DailyCrossSectionDataset(Dataset):
    def __init__(self, X, y, di, rows):
        self.X = X
        self.y = y
        rows = np.asarray(rows, dtype=np.int64)
        d = di[rows]

        order = np.argsort(d, kind='mergesort')
        rows_sorted = rows[order]
        d_sorted = d[order]

        uniq, starts = np.unique(d_sorted, return_index=True)
        starts = list(starts) + [len(rows_sorted)]

        self.groups = []
        for i in tqdm(range(len(uniq)), desc='Build day groups', leave=False):
            s, e = starts[i], starts[i + 1]
            self.groups.append(rows_sorted[s:e])

    def __len__(self):
        return len(self.groups)

    def __getitem__(self, idx):
        r = self.groups[idx]
        x = torch.from_numpy(self.X[r]).float()          # (N_stocks, Features)
        if self.y is None:
            y = torch.zeros((len(r),), dtype=torch.float32)
        else:
            y = torch.from_numpy(self.y[r]).float()      # (N_stocks,)
        ridx = torch.from_numpy(r).long()
        return x, y, ridx


class CrossSectionalTransformer(nn.Module):
    def __init__(self, in_dim, d_model=64, nhead=4, nlayers=2, dropout=0.1):
        super().__init__()
        self.in_proj = nn.Linear(in_dim, d_model)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 2,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=nlayers)
        self.out = nn.Linear(d_model, 1)

    def forward(self, x):
        # x: (N_stocks, Features) or (1, N_stocks, Features)
        if x.dim() == 2:
            x = x.unsqueeze(0)
        h = self.in_proj(x)
        h = self.encoder(h)
        s = self.out(h).squeeze(-1)
        return s.squeeze(0)  # (N_stocks,)


class PairwiseMarginRankingLoss(nn.Module):
    def __init__(self, margin=0.01, max_pairs=5000):
        super().__init__()
        self.margin = margin
        self.max_pairs = max_pairs

    def forward(self, pred, target):
        n = pred.numel()
        if n < 2:
            return pred.new_tensor(0.0, requires_grad=True)

        draw = min(max(self.max_pairs * 4, self.max_pairs), n * n)
        i = torch.randint(0, n, (draw,), device=pred.device)
        j = torch.randint(0, n, (draw,), device=pred.device)
        m = target[i] > target[j]

        if not m.any():
            return pred.new_tensor(0.0, requires_grad=True)

        i = i[m][:self.max_pairs]
        j = j[m][:self.max_pairs]

        if i.numel() == 0:
            return pred.new_tensor(0.0, requires_grad=True)

        y = torch.ones_like(pred[i])
        return F.margin_ranking_loss(pred[i], pred[j], y, margin=self.margin)


def predict_daily_model(model, loader, n_rows):
    model.eval()
    out = np.zeros(n_rows, dtype=np.float32)
    with torch.no_grad():
        for xb, _, ridx in tqdm(loader, desc='Transformer infer', leave=False):
            xb = xb.squeeze(0).to(DEVICE, non_blocking=True)
            ridx = ridx.squeeze(0).numpy()
            p = model(xb).detach().cpu().numpy().astype(np.float32)
            out[ridx] = p
    return out


def train_transformer_fold(X, y, di, train_rows, val_rows, test_X, test_di):
    tr_ds = DailyCrossSectionDataset(X, y, di, train_rows)
    va_ds = DailyCrossSectionDataset(X, y, di, val_rows)
    te_ds = DailyCrossSectionDataset(test_X, None, test_di, np.arange(len(test_X)))

    tr_dl = DataLoader(tr_ds, batch_size=1, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    va_dl = DataLoader(va_ds, batch_size=1, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    te_dl = DataLoader(te_ds, batch_size=1, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

    model = CrossSectionalTransformer(
        in_dim=X.shape[1],
        d_model=TRF_D_MODEL,
        nhead=TRF_HEADS,
        nlayers=TRF_LAYERS,
        dropout=TRF_DROPOUT,
    ).to(DEVICE)

    crit = PairwiseMarginRankingLoss(margin=0.01, max_pairs=PAIR_MAX)
    opt = torch.optim.AdamW(model.parameters(), lr=TRF_LR, weight_decay=TRF_WD)

    model.train()
    for ep in tqdm(range(TRF_EPOCHS), desc='Transformer epochs', leave=False):
        losses = []
        for xb, yb, _ in tqdm(tr_dl, desc=f'Transformer train e{ep+1}', leave=False):
            xb = xb.squeeze(0).to(DEVICE, non_blocking=True)
            yb = yb.squeeze(0).to(DEVICE, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            pred = model(xb)
            loss = crit(pred, yb)
            if not torch.isfinite(loss):
                continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            losses.append(loss.item())

        print(f'ep={ep+1} pairwise_loss={np.mean(losses) if losses else np.nan:.6f}')
        cleanup()

    val_pred_all = predict_daily_model(model, va_dl, len(X))
    test_pred = predict_daily_model(model, te_dl, len(test_X))

    del tr_ds, va_ds, te_ds, tr_dl, va_dl, te_dl
    cleanup()
    return model, val_pred_all[val_rows], test_pred


In [ ]:
# ============================================================
# 5) PHASE 4: LightGBM + Transformer OOF + SLSQP blend
# ============================================================
folds = make_walkforward_folds(di_train, n_folds=N_FOLDS, embargo=EMBARGO)
print('fold count:', len(folds))

X_train_final = X_train_neut.astype(np.float32)
X_test_final = X_test_neut.astype(np.float32)

oof_lgb = np.zeros(len(train), dtype=np.float32)
oof_trf = np.zeros(len(train), dtype=np.float32)
test_lgb = np.zeros(len(test), dtype=np.float32)
test_trf = np.zeros(len(test), dtype=np.float32)

lgb_params = {
    'objective': 'regression',
    'metric': 'l2',
    'learning_rate': 0.03 if FAST_MODE else 0.02,
    'num_leaves': 63,
    'feature_fraction': 0.85,
    'bagging_fraction': 0.85,
    'bagging_freq': 1,
    'min_data_in_leaf': 64,
    'lambda_l1': 0.0,
    'lambda_l2': 1.0,
    'num_threads': -1,
    'seed': SEED,
    'verbosity': -1,
}

for fidx, (tr_dates, va_dates) in enumerate(tqdm(folds, desc='CV folds')):
    tr_idx = np.where(np.isin(di_train, tr_dates))[0]
    va_idx = np.where(np.isin(di_train, va_dates))[0]

    if len(tr_idx) == 0 or len(va_idx) == 0:
        continue

    # LightGBM
    dtr = lgb.Dataset(X_train_final[tr_idx], label=y_train[tr_idx], free_raw_data=True)
    dva = lgb.Dataset(X_train_final[va_idx], label=y_train[va_idx], free_raw_data=True)

    lgb_model = lgb.train(
        lgb_params,
        dtr,
        num_boost_round=600 if FAST_MODE else 1200,
        valid_sets=[dva],
        callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)],
    )

    oof_lgb[va_idx] = lgb_model.predict(X_train_final[va_idx], num_iteration=lgb_model.best_iteration)
    test_lgb += lgb_model.predict(X_test_final, num_iteration=lgb_model.best_iteration) / len(folds)

    # Transformer fold
    trf_model, val_trf_pred, te_trf_pred = train_transformer_fold(
        X_train_final, y_train, di_train,
        tr_idx, va_idx,
        X_test_final, di_test
    )

    oof_trf[va_idx] = val_trf_pred
    test_trf += te_trf_pred / len(folds)

    lgb_ic = date_ic(y_train[va_idx], oof_lgb[va_idx], di_train[va_idx])
    trf_ic = date_ic(y_train[va_idx], oof_trf[va_idx], di_train[va_idx])
    print(f'fold={fidx+1} lgb_ic={lgb_ic:.6f} trf_ic={trf_ic:.6f}')

    del dtr, dva, lgb_model, trf_model
    cleanup()


# Blend optimization (maximize OOF IC)
valid_mask = np.isfinite(oof_lgb) & np.isfinite(oof_trf) & np.isfinite(y_train)

def blend_objective(w):
    p = w[0] * oof_lgb[valid_mask] + w[1] * oof_trf[valid_mask]
    return -date_ic(y_train[valid_mask], p, di_train[valid_mask])

cons = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0},)
bounds = [(0.0, 1.0), (0.0, 1.0)]
res = minimize(blend_objective, x0=np.array([0.5, 0.5]), method='SLSQP', bounds=bounds, constraints=cons)

if res.success:
    w_opt = res.x.astype(np.float64)
else:
    w_opt = np.array([0.5, 0.5], dtype=np.float64)

print('optimal_weights:', w_opt)

oof_blend = w_opt[0] * oof_lgb + w_opt[1] * oof_trf
test_blend = w_opt[0] * test_lgb + w_opt[1] * test_trf

print('OOF IC LGB :', date_ic(y_train, oof_lgb, di_train))
print('OOF IC TRF :', date_ic(y_train, oof_trf, di_train))
print('OOF IC BLEND:', date_ic(y_train, oof_blend, di_train))

submission = pd.DataFrame({
    'id': test['id'] if 'id' in test.columns else np.arange(len(test), dtype=np.int64),
    'target': np.where(np.isfinite(test_blend), test_blend, 0.0).astype(np.float64),
})

if (np.abs(submission['target'].values) > 0).mean() < 0.10:
    submission['target'] += np.random.normal(0, 1e-8, len(submission))

submission.to_csv(SUB_PATH, index=False)
print('saved:', SUB_PATH, submission.shape)
cleanup()


### Notes
- This notebook uses strict per-date neutralization and walk-forward date splits to avoid leakage.
- All heavy loops use `tqdm`, and memory cleanup is called aggressively after major stages/folds/epochs.
- Transformer attention is across stocks within each date, not across time.
